Environment: Harpy

### Import packages

In [1]:
import anndata as ad
import harpy as hp
import os
import numpy as np
import pandas as pd
import geopandas as gpd
from skimage import io, morphology
from matplotlib import pyplot as plt
import gc 

### Specify paths

In [2]:
path_hne_image = r'e:\2024-07-UTBenchmark-SpC\data\processed\Nova-ST\Exp44\input\S1_RGB.tif'
path_transcripts = r'e:\2024-07-UTBenchmark-SpC\data\processed\Nova-ST\Exp44\novascope_output_with_histology\analysis\novaseq6000-20240304-b04d-mouse-ae2e1\novaseq6000-20240304-b04d-mouse-ae2e1-default\preprocess\novaseq6000-20240304-b04d-mouse-ae2e1-default.transcripts.tsv.gz'
output_path = r'E:\Nova-ST\output_exp44'
plots_path = os.path.join(output_path, 'plots')
os.makedirs(output_path, exist_ok=True)
os.makedirs(plots_path, exist_ok=True)

In [ ]:
# import spatialdata as sd
# sdata = sd.read_zarr(r'e:\Nova-ST\output_exp44\sdata.zarr')

c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\xarray_schema\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
version mismatch: detected: RasterFormatV02, requested: FormatV04
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\zarr\creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
version mismatch: detected: RasterFormatV02, requested: FormatV04


### Create sdata

In [4]:
hne_img_layer = "hne_image_rgb"

sdata = hp.io.create_sdata(
    input=path_hne_image,
    output_path=os.path.join(output_path, "sdata.zarr"),
    img_layer=hne_img_layer,
    chunks=1024,
    scale_factors=(2, 2, 2, 2),
)
sdata

c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\zarr\creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)


INFO     The Zarr backing store has been changed from None the new file path: E:\Nova-ST\output_exp44\sdata.zarr   


2025-08-28 17:03:02,600 - harpy.image._manager - INFO - Writing results to layer 'hne_image_rgb'
version mismatch: detected: RasterFormatV02, requested: FormatV04
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\zarr\creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)


SpatialData object, with associated Zarr store: E:\Nova-ST\output_exp44\sdata.zarr
└── Images
      └── 'hne_image_rgb': DataTree[cyx] (3, 10194, 14049), (3, 5097, 7024), (3, 2548, 3512), (3, 1274, 1756), (3, 637, 878)
with coordinate systems:
    ▸ 'global', with elements:
        hne_image_rgb (Images)

In [5]:
# Plot H&E
import spatialdata_plot
import xarray
xa: xarray.DataArray = sdata.images[hne_img_layer][f'scale0']['image']
shape = (xa.shape[1], xa.shape[2])

figsize = (shape[1]/800, shape[0]/800)
fig, ax = plt.subplots(figsize=figsize, dpi=300)
ax.set_xlabel("x", fontsize=12)
ax.set_ylabel("y", fontsize=12)

from matplotlib.colors import Normalize

sdata.pl.render_images(
    element=hne_img_layer
).pl.show(
    coordinate_systems="global",
    title="H&E",
    figsize=figsize,
    dpi=300,
    colorbar=False,
    ax=ax,
)

save_path = os.path.join(output_path, 'plots', f'HE.png')
fig.savefig(save_path, bbox_inches="tight", facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

INFO     Rasterizing image for faster rendering.                                                                   


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.0..1.1538461].


Plot saved at: E:\Nova-ST\output_exp44\plots\HE.png


In [6]:
sdata = hp.io.read_transcripts(
    sdata,
    path_count_matrix=path_transcripts,
    column_x=2,
    column_y=3,
    column_z=None,
    column_gene=5,       # column 5 is the "readable" gene name, column 4 is the Ensemble Gene ID
    column_midcount=6,
    delimiter='\t',
    header=0,
    comment=None,
    transform_matrix=np.array([[0.001, 0, 0],[0, 0.001, 0],[0, 0, 1]]),
    overwrite=True
)
sdata

c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\dask\dataframe\io\csv.py:555: UserWarning: Warning gzip compression does not support breaking apart files
Please ensure that each individual file can fit in memory and
use the keyword ``blocksize=None to remove this message``
Setting ``blocksize=None``
  warn(
2025-08-28 17:03:38,775 - harpy.io._transcripts - INFO - Transform matrix used:
 [[0.001 0.    0.   ]
 [0.    0.001 0.   ]
 [0.    0.    1.   ]]


SpatialData object, with associated Zarr store: E:\Nova-ST\output_exp44\sdata.zarr
├── Images
│     └── 'hne_image_rgb': DataTree[cyx] (3, 10194, 14049), (3, 5097, 7024), (3, 2548, 3512), (3, 1274, 1756), (3, 637, 878)
└── Points
      └── 'transcripts': DataFrame with shape: (<Delayed>, 3) (2D points)
with coordinate systems:
    ▸ 'global', with elements:
        hne_image_rgb (Images), transcripts (Points)

### Plot transcript density

In [7]:
sdata = hp.im.transcript_density(
    sdata,
    img_layer='hne_image_rgb',
    points_layer='transcripts',
    output_layer='transcript_density',
    overwrite=True
)
sdata

2025-08-28 17:10:12,878 - harpy.image._transcripts - INFO - The number of transcripts ( 67623012 ) is larger than n_sample, sampling 15000000 transcripts.
2025-08-28 17:10:12,908 - harpy.image._transcripts - INFO - sampling finished
2025-08-28 17:10:13,766 - harpy.image._manager - INFO - Writing results to layer 'transcript_density'
version mismatch: detected: RasterFormatV02, requested: FormatV04
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\zarr\creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
version mismatch: detected: RasterFormatV02, requested: FormatV04


SpatialData object, with associated Zarr store: E:\Nova-ST\output_exp44\sdata.zarr
├── Images
│     ├── 'hne_image_rgb': DataTree[cyx] (3, 10194, 14049), (3, 5097, 7024), (3, 2548, 3512), (3, 1274, 1756), (3, 637, 878)
│     └── 'transcript_density': DataArray[cyx] (1, 10194, 14049)
└── Points
      └── 'transcripts': DataFrame with shape: (<Delayed>, 3) (2D points)
with coordinate systems:
    ▸ 'global', with elements:
        hne_image_rgb (Images), transcript_density (Images), transcripts (Points)

In [8]:
hp.pl.transcript_density(
    sdata,
    img_layer=[hne_img_layer, "transcript_density"],
    channel=0,
    output=os.path.join(plots_path, "transcript_density.png")
)

### Add hexagonal grids

In [9]:
# Create hexagonal grids
for lvl in [10, 20, 40, 100]:
    sdata = hp.im.add_grid_labels_layer(
        sdata, 
        shape=shape, 
        size=lvl/2,
        output_shapes_layer=f"shapes_spots_{lvl}um", 
        output_labels_layer=f"labels_spots_{lvl}um", 
        grid_type='hexagon', 
        offset=(0, 0), 
        chunks=1024, 
        client=None, 
        transformations=None, 
        scale_factors=(2, 2, 2, 2),
        overwrite=True
    )

2025-08-28 17:16:31,820 - harpy.image._manager - INFO - Writing results to layer 'labels_spots_10um'
2025-08-28 17:21:44,562 - harpy.image._manager - INFO - Writing results to layer 'labels_spots_20um'
2025-08-28 17:23:38,428 - harpy.image._manager - INFO - Writing results to layer 'labels_spots_40um'
2025-08-28 17:24:23,044 - harpy.image._manager - INFO - Writing results to layer 'labels_spots_100um'


In [9]:
# Plot hexagonal grid labels
for lvl in [10, 20, 40, 100]:  
    
    plots_path_lvl = os.path.join(plots_path, f'{lvl}um')
    os.makedirs(plots_path_lvl, exist_ok=True)
    
    figsize = (shape[1]/150, shape[0]/150)
    fig, ax = plt.subplots(figsize=figsize, dpi=300)
    ax.set_xlabel("x", fontsize=12)
    ax.set_ylabel("y", fontsize=12)

    sdata.pl.render_images(
        element=hne_img_layer
    ).pl.render_labels(
        element=f"labels_spots_{lvl}um", fill_alpha=0.1, contour_px=5, outline_alpha=1
    ).pl.show(
        coordinate_systems="global",
        figsize=figsize,
        dpi=300,
        colorbar=False,
        ax=ax,
    )

    save_path = os.path.join(plots_path_lvl, f'plot_grid_{lvl}um.png')
    fig.savefig(save_path, bbox_inches="tight", facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path)

MemoryError: Unable to allocate 18.9 GiB for an array with shape (22012, 28790, 4) and data type float64

Error in callback <function _draw_all_if_interactive at 0x00000214755A9F30> (for post_execute), with arguments args (),kwargs {}:


MemoryError: Unable to allocate 9.44 GiB for an array with shape (22012, 28790, 4) and data type float32

MemoryError: Unable to allocate 9.44 GiB for an array with shape (22012, 28790, 4) and data type float32

<Figure size 37390x28588 with 1 Axes>

In [10]:
import gc
gc.collect()

113159

In [11]:
# Create tables
for lvl in [10, 20, 40, 100]:  
    sdata = hp.tb.allocate(
        sdata=sdata,
        labels_layer=f"labels_spots_{lvl}um",
        points_layer="transcripts",
        output_layer=f"{lvl}um",
        update_shapes_layers=False,
        overwrite=True,
    )

2025-08-28 17:24:32,745 - harpy.table._allocation - INFO - Calculating cell counts.


2025-08-28 17:27:05,990 - harpy.table._allocation - INFO - Calculating cell counts.
2025-08-28 17:28:58,153 - harpy.table._allocation - INFO - Calculating cell counts.
2025-08-28 17:30:27,400 - harpy.table._allocation - INFO - Calculating cell counts.


### Add tissue mask

In [27]:
import geopandas as gpd
gdf = gpd.read_file(r'e:\Nova-ST\tissue_annotation\tissue.geojson')
gdf.index = gdf.index + 1

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gdf,
    output_layer = 'tissue',
    overwrite = True,
)

sdata = hp.im.rasterize(
    sdata,
    shapes_layer = 'tissue',
    output_layer = 'tissue_mask',
    chunks = 1024,
    overwrite = True,
)

2025-07-18 11:43:08,519 - harpy.utils._io - WARNING - layer with name 'tissue' already exists. Overwriting...
2025-07-18 11:43:14,926 - harpy.image._manager - INFO - Writing results to layer 'tissue_mask'
c:\Users\julien.mortier\conda-envs\training_env\Lib\site-packages\zarr\creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
2025-07-18 11:43:44,449 - harpy.utils._io - WARNING - layer with name 'tissue_mask' already exists. Overwriting...


In [8]:
gdf = sdata.shapes['Shapes']
gdf.index = gdf.index + 1

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gdf,
    output_layer = 'tissue',
    overwrite = True,
)

sdata = hp.im.rasterize(
    sdata,
    shapes_layer = 'tissue',
    output_layer = 'tissue_mask',
    chunks = 1024,
    overwrite = True,
)

2025-08-29 10:33:53,177 - harpy.image._manager - INFO - Writing results to layer 'tissue_mask'


In [ ]:
# from napari_spatialdata import Interactive
# Interactive(sdata)

2025-07-18 11:54:14.610 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
2025-07-18 11:54:15.418 | DEBUG    | napari_spatialdata._view:_on_layer_update:569 - Updating layer.
c:\Users\julien.mortier\conda-envs\training_env\Lib\site-packages\anndata\_core\aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


### Filter out spots not in tissue

In [9]:
sdata

SpatialData object, with associated Zarr store: E:\Nova-ST\output_exp44\sdata.zarr
├── Images
│     ├── 'hne_image_rgb': DataTree[cyx] (3, 10194, 14049), (3, 5097, 7024), (3, 2548, 3512), (3, 1274, 1756), (3, 637, 878)
│     └── 'transcript_density': DataArray[cyx] (1, 10194, 14049)
├── Labels
│     ├── 'labels_spots_10um': DataTree[yx] (10194, 14049), (5097, 7024), (2548, 3512), (1274, 1756), (637, 878)
│     ├── 'labels_spots_20um': DataTree[yx] (10194, 14049), (5097, 7024), (2548, 3512), (1274, 1756), (637, 878)
│     ├── 'labels_spots_40um': DataTree[yx] (10194, 14049), (5097, 7024), (2548, 3512), (1274, 1756), (637, 878)
│     ├── 'labels_spots_100um': DataTree[yx] (10194, 14049), (5097, 7024), (2548, 3512), (1274, 1756), (637, 878)
│     └── 'tissue_mask': DataArray[yx] (6237, 5352)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 3) (2D points)
├── Shapes
│     ├── 'Shapes': GeoDataFrame shape: (1, 1) (2D shapes)
│     ├── 'shapes_spots_10um': GeoDataFrame s

In [10]:
def add_in_tissue(sdata, table_layer, tissue_mask):
    def assign_region(centroid, mask, mask_shape):
        x, y = int(round(centroid[0])), int(round(centroid[1]))
        
        if 0 <= y < mask_shape[0] and 0 <= x < mask_shape[1]:
            return 1 if mask[y, x] == 1 else 0
        return 0  # Outside mask bounds

    adata = sdata.tables[table_layer].copy()
    coordinates = adata.obsm['spatial']
    coordinates_df = pd.DataFrame(coordinates, columns=["x", "y"], index=adata.obs.index)

    mask_shape = tissue_mask.shape
    adata.obs["in_tissue"] = coordinates_df.apply(
        lambda row: assign_region((row["x"], row["y"]), tissue_mask, mask_shape), axis=1
    )

    sdata = hp.tb.add_table_layer(
        sdata,
        adata=adata.copy(),
        output_layer=table_layer,
        region=[f"labels_spots_{table_layer}"],
        overwrite=True,
    )
    
    adata_in_tissue = adata[adata.obs["in_tissue"] == 1].copy()
    
    sdata = hp.tb.add_table_layer(
        sdata,
        adata=adata_in_tissue.copy(),
        output_layer=f'{table_layer}_in_tissue',
        region=[f"labels_spots_{table_layer}"],
        overwrite=True,
    )

    return sdata


In [ ]:
for lvl in [10, 20, 40, 100]:
    sdata = add_in_tissue(
        sdata, 
        table_layer = f'{lvl}um', 
        tissue_mask = sdata.labels['tissue_mask'].compute().data
    )

2025-08-29 10:36:44,425 - harpy.utils._io - WARNING - layer with name '10um' already exists. Overwriting...


### Leiden clustering

In [ ]:
for lvl in [20, 40, 100]:
    print(f'running: {lvl}um')

    labels_layer=f"labels_spots_{lvl}um"
    shapes_layer=f"shapes_spots_{lvl}um"
    table_layer=f"{lvl}um_in_tissue"
    
    plots_path_lvl = os.path.join(plots_path, f'{lvl}um')
    os.makedirs(plots_path_lvl, exist_ok=True)
    
    # Preprocess data
    sdata = hp.tb.preprocess_transcriptomics(
        sdata=sdata,
        size_norm=True,
        labels_layer=labels_layer,
        table_layer=table_layer,
        output_layer=f"{table_layer}_in_tissue_processed",
        min_counts=10,
        min_cells=5,
        highly_variable_genes=True,
        max_value_scale=10,
        n_comps = 50,
        update_shapes_layers=False,
        overwrite=True
    )
    
    gc.collect()
    
    # Leiden clustering
    leiden_resolutions = [0.6]

    for res in leiden_resolutions:
        res_str = str(res).replace('.', '')
                
        sdata = hp.tb.leiden(
            sdata, 
            labels_layer=labels_layer,
            table_layer=f"{table_layer}_in_tissue_processed",
            output_layer=f"{table_layer}_in_tissue_processed",
            calculate_umap=True,
            calculate_neighbors=True,
            rank_genes=True,
            n_neighbors=35,
            n_pcs=30,
            resolution=res,
            key_added=f'leiden_{res_str}',
            index_names_var=None,
            index_positions_var=None,
            random_state=100,
            overwrite=True
        )
        
        hp.pl.cluster(
            sdata, 
            table_layer = f"{table_layer}_in_tissue_processed",
            key_added=f'leiden_{res_str}',
            output=os.path.join(plots_path_lvl, f'UMAP_leiden_in_tissue_{res_str}')
        )
        
        figsize = (shape[1]/400, shape[0]/400)
        fig, ax = plt.subplots(figsize=figsize, dpi=300)
        ax.set_xlabel("x", fontsize=12)
        ax.set_ylabel("y", fontsize=12)

        sdata.pl.render_images(
            element=hne_img_layer
        ).pl.render_labels(
            element=labels_layer,
            color=f'leiden_{res_str}',
            table_name=f"{table_layer}_in_tissue_processed",
            cmap="rainbow",
            fill_alpha=1, 
            contour_px=1, 
            outline_alpha=0
        ).pl.show(
            coordinate_systems="global",
            title=f"Leiden {res_str}",
            figsize=figsize,
            dpi=300,
            colorbar=True,
            ax=ax,
        )

        save_path = os.path.join(plots_path_lvl, f'leiden_{res_str}_in_tissue.png')
        fig.savefig(save_path, bbox_inches="tight", facecolor="white", transparent=False)

        plt.close(fig)

        print("Plot saved at:", save_path)
    
        gc.collect()

running: 20um


2025-08-28 18:06:02,716 - harpy.table._preprocess - INFO - Calculating cell size from provided labels_layer 'labels_spots_20um'
2025-08-28 18:20:45,654 - harpy.utils._io - WARNING - layer with name '20um_processed' already exists. Overwriting...


INFO     Rasterizing image for faster rendering.                                                                   


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.0..1.0851064].


INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: E:\Nova-ST\output_exp44\plots\20um\leiden_06.png


### Plot QC metrics

In [ ]:
for lvl in [20, 40, 100]:
    labels_layer=f"labels_spots_{lvl}um"
    shapes_layer=f"shapes_spots_{lvl}um"
    table_layer = f'{lvl}um_in_tissue_processed'
    
    plots_path_lvl = os.path.join(plots_path, f'{lvl}um')
    os.makedirs(plots_path_lvl, exist_ok=True)
    
    for col in ['total_counts', 'n_genes_by_counts']:
        figsize = (shape[1]/400, shape[0]/400)
        fig, ax = plt.subplots(figsize=figsize, dpi=300)
        ax.set_xlabel("x", fontsize=12)
        ax.set_ylabel("y", fontsize=12)

        sdata.pl.render_images(
            element=hne_img_layer
        ).pl.render_labels(
            element=labels_layer,
            color=col,
            table_name=table_layer,
            cmap="viridis",
            fill_alpha=1, 
            contour_px=1, 
            outline_alpha=0
        ).pl.show(
            coordinate_systems="global",
            title=col,
            figsize=figsize,
            dpi=300,
            colorbar=True,
            ax=ax,
        )

        save_path = os.path.join(plots_path_lvl, f'{col}_{table_layer}_in_tissue.png')
        fig.savefig(save_path, bbox_inches="tight", facecolor="white", transparent=False)

        plt.close(fig)

        print("Plot saved at:", save_path)
        
        gc.collect()

INFO     Rasterizing image for faster rendering.                                                                   


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.0..1.0851064].


INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: E:\Nova-ST\output_exp44\plots\20um\total_counts_20um_processed.png
INFO     Rasterizing image for faster rendering.                                                                   


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [0.0..1.0851064].


INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: E:\Nova-ST\output_exp44\plots\20um\n_genes_by_counts_20um_processed.png


In [6]:
from spatialdata import read_zarr

sdata.write_element(element_name='Shapes')
sdata = read_zarr(sdata.path)
sdata

version mismatch: detected: RasterFormatV02, requested: FormatV04
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\zarr\creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
version mismatch: detected: RasterFormatV02, requested: FormatV04


SpatialData object, with associated Zarr store: E:\Nova-ST\output_exp44\sdata.zarr
├── Images
│     ├── 'hne_image_rgb': DataTree[cyx] (3, 10194, 14049), (3, 5097, 7024), (3, 2548, 3512), (3, 1274, 1756), (3, 637, 878)
│     └── 'transcript_density': DataArray[cyx] (1, 10194, 14049)
├── Labels
│     ├── 'labels_spots_10um': DataTree[yx] (10194, 14049), (5097, 7024), (2548, 3512), (1274, 1756), (637, 878)
│     ├── 'labels_spots_20um': DataTree[yx] (10194, 14049), (5097, 7024), (2548, 3512), (1274, 1756), (637, 878)
│     ├── 'labels_spots_40um': DataTree[yx] (10194, 14049), (5097, 7024), (2548, 3512), (1274, 1756), (637, 878)
│     └── 'labels_spots_100um': DataTree[yx] (10194, 14049), (5097, 7024), (2548, 3512), (1274, 1756), (637, 878)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 3) (2D points)
├── Shapes
│     ├── 'Shapes': GeoDataFrame shape: (1, 1) (2D shapes)
│     ├── 'shapes_spots_10um': GeoDataFrame shape: (2201997, 1) (2D shapes)
│     ├── 'shapes_spo